In [1]:
!pip install tensorboardX rdkit
!pip install rdkit-pypi
import os
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as Data

from sklearn.model_selection import LeaveOneOut
import time
from tqdm import tqdm
import numpy as np
import gc
import sys
sys.setrecursionlimit(50000)
import pickle
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
# from tensorboardX import SummaryWriter
torch.nn.Module.dump_patches = True
import copy
import pandas as pd
#then import my own modules
from GCN import save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight 
from rdkit import Chem
# from rdkit.Chem import AllChem
from rdkit.Chem import QED
from rdkit.Chem import rdMolDescriptors, MolSurf
from rdkit.Chem.Draw import SimilarityMaps
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
%matplotlib inline
from numpy.polynomial.polynomial import polyfit
import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib.cm as cm
import matplotlib
import seaborn as sns; sns.set_style("darkgrid")
from IPython.display import SVG, display
from torch.utils.data import Dataset, TensorDataset, DataLoader

import random
import itertools
from sklearn.metrics import r2_score
import scipy
from itertools import combinations
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import pickle
import os
import time
from rdkit import Chem
from sklearn.model_selection import LeaveOneOut
from collections import defaultdict


device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:451.)
  _C._set_default_tensor_type(t)


cuda


In [2]:

class Fingerprint(nn.Module):
    def __init__(self, radius, T, input_feature_dim, input_bond_dim,
                 fingerprint_dim, output_units_num, p_dropout):
        super(Fingerprint, self).__init__()
        
        self.radius = radius
        self.T = T
        
        # Atom embedding
        self.atom_fc = nn.Linear(input_feature_dim, fingerprint_dim)
        self.neighbor_fc = nn.Linear(input_feature_dim + input_bond_dim, fingerprint_dim)
        self.gru_cells = nn.ModuleList([nn.GRUCell(fingerprint_dim, fingerprint_dim) for _ in range(radius)])
        self.align_layers = nn.ModuleList([nn.Linear(2 * fingerprint_dim, 1) for _ in range(radius)])
        self.attend_layers = nn.ModuleList([nn.Linear(fingerprint_dim, fingerprint_dim) for _ in range(radius)])
        
        # Molecule embedding
        self.mol_gru_cell = nn.GRUCell(fingerprint_dim, fingerprint_dim)
        self.mol_align = nn.Linear(2 * fingerprint_dim, 1)
        self.mol_attend = nn.Linear(fingerprint_dim, fingerprint_dim)
        
        self.dropout = nn.Dropout(p=p_dropout)
        self.output = nn.Linear(fingerprint_dim, output_units_num)
        self.pairwise_output = nn.Linear(fingerprint_dim * 2, output_units_num)
        
    def forward(self, atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1,
                atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2):
        # Process first molecule
        atom_feature1, mol_feature1 = self.process_single_molecule(
            atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1
        )

        # Process second molecule
        atom_feature2, mol_feature2 = self.process_single_molecule(
            atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2
        )

        # Concatenate molecular features
        combined_feature = torch.cat([mol_feature1, mol_feature2], dim=1)

        # Final pairwise prediction
        pairwise_prediction = self.pairwise_output(self.dropout(combined_feature))

        return atom_feature1, atom_feature2, pairwise_prediction

    def process_single_molecule(self, atom_list, bond_list, atom_degree_list, bond_degree_list, atom_mask):
        batch_size, mol_length, num_atom_feat = atom_list.size()
        atom_mask = atom_mask.unsqueeze(2)

        atom_feature = F.leaky_relu(self.atom_fc(atom_list))
        neighbor_feature = self.get_neighbor_feature(atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size)

        attend_mask, softmax_mask = self.create_masks(atom_degree_list, mol_length)

        atom_feature = self.apply_atom_attention(atom_feature, neighbor_feature, attend_mask, softmax_mask)
        mol_feature = torch.sum(F.relu(atom_feature) * atom_mask, dim=-2)

        mol_feature = self.apply_molecule_attention(atom_feature, mol_feature, atom_mask)

        return atom_feature, mol_feature


    def get_neighbor_feature(self, atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size):
        bond_neighbor = torch.stack([bond_list[i][bond_degree_list[i]] for i in range(batch_size)], dim=0)
        atom_neighbor = torch.stack([atom_list[i][atom_degree_list[i]] for i in range(batch_size)], dim=0)
        neighbor_feature = torch.cat([atom_neighbor, bond_neighbor], dim=-1)
        return F.leaky_relu(self.neighbor_fc(neighbor_feature))

    def create_masks(self, atom_degree_list, mol_length):
        attend_mask = (atom_degree_list != mol_length - 1).float().unsqueeze(-1)
        softmax_mask = torch.where(atom_degree_list == mol_length - 1, torch.tensor(-9e8).cuda(), torch.tensor(0.).cuda()).unsqueeze(-1)
        return attend_mask, softmax_mask

    def apply_atom_attention(self, atom_feature, neighbor_feature, attend_mask, softmax_mask):
        #print("Applying atom-level attention")
        batch_size, mol_length = atom_feature.shape[:2]
        
        for d in range(self.radius):
            atom_feature_expand = atom_feature.unsqueeze(-2).expand(-1, -1, neighbor_feature.size(2), -1)
            feature_align = torch.cat([atom_feature_expand, neighbor_feature], dim=-1)
            
            align_score = F.leaky_relu(self.align_layers[d](self.dropout(feature_align))) + softmax_mask
            attention_weight = F.softmax(align_score, -2) * attend_mask
            
            context = torch.sum(attention_weight * self.attend_layers[d](self.dropout(neighbor_feature)), -2)
            context = F.elu(context)
            
            atom_feature = self.gru_cells[d](
                context.view(batch_size * mol_length, -1),
                atom_feature.view(batch_size * mol_length, -1)
            ).view(batch_size, mol_length, -1)
            
            neighbor_feature = F.relu(atom_feature).unsqueeze(-2).expand(-1, -1, neighbor_feature.size(2), -1)
        
        return atom_feature

    def apply_molecule_attention(self, atom_feature, mol_feature, atom_mask):
        #print("Applying mol-level attention")
        batch_size, mol_length = atom_feature.shape[:2]
        mol_softmax_mask = torch.where(atom_mask.squeeze() == 0, torch.tensor(-9e8).cuda(), torch.tensor(0.).cuda())
        
        for _ in range(self.T):
            mol_prediction_expand = mol_feature.unsqueeze(-2).expand(-1, mol_length, -1)
            mol_align = torch.cat([mol_prediction_expand, atom_feature], dim=-1)
            mol_align_score = F.leaky_relu(self.mol_align(mol_align)) + mol_softmax_mask.unsqueeze(-1)
            mol_attention_weight = F.softmax(mol_align_score, -2) * atom_mask
            
            mol_context = torch.sum(mol_attention_weight * self.mol_attend(self.dropout(atom_feature)), -2)
            mol_context = F.elu(mol_context)
            mol_feature = self.mol_gru_cell(mol_context, mol_feature)
        
        return mol_feature


class RelativeGCNModel(nn.Module):
    def __init__(self, radius, T, input_feature_dim, input_bond_dim,
                 fingerprint_dim, output_units_num, p_dropout):
        super(RelativeGCNModel, self).__init__()
        
        self.radius = radius
        self.T = T
        

        hidden_dim = fingerprint_dim * 4  
        self.atom_fc = nn.Sequential(
            nn.Linear(input_feature_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, fingerprint_dim)
        )
        
        self.neighbor_fc = nn.Sequential(
            nn.Linear(input_feature_dim + input_bond_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, fingerprint_dim)
        )
        

        self.gcn_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fingerprint_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, fingerprint_dim)
            ) for _ in range(radius * 2)  
        ])
        
        self.dropout = nn.Dropout(p=p_dropout)
        
        # More complex output layer for pairwise prediction
        self.output = nn.Sequential(
            nn.Linear(fingerprint_dim * 2, hidden_dim),  # Input is concatenated features of two molecules
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_units_num)
        )

        
    def forward(self, atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1,
                atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2):
        
        batch_size, mol_length, num_atom_feat = atom_list1.size()

        # Process first molecule
        atom_feature1, mol_feature1 = self.process_molecule(atom_list1, bond_list1, atom_degree_list1, bond_degree_list1, atom_mask1)
        
        # Process second molecule
        atom_feature2, mol_feature2 = self.process_molecule(atom_list2, bond_list2, atom_degree_list2, bond_degree_list2, atom_mask2)
        
        # Concatenate features of both molecules and predict pairwise difference
        pairwise_feature = torch.cat([mol_feature1, mol_feature2], dim=1)
        pairwise_prediction = self.output(self.dropout(pairwise_feature))
        
        return atom_feature1, atom_feature2, pairwise_prediction

    
    
    def process_molecule(self, atom_list, bond_list, atom_degree_list, bond_degree_list, atom_mask):
        batch_size, mol_length, num_atom_feat = atom_list.size()
        atom_mask = atom_mask.unsqueeze(2)
        

        atom_feature = self.atom_fc(atom_list)
        
        neighbor_feature = self.get_neighbor_feature(atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size, mol_length)
        
        for i, layer in enumerate(self.gcn_layers):
            neighbor_feature = layer(neighbor_feature)
            atom_feature = atom_feature + neighbor_feature
            atom_feature = self.dropout(atom_feature)
        
        mol_feature = torch.sum(F.relu(atom_feature) * atom_mask, dim=1)
        return atom_feature, mol_feature

    

    def get_neighbor_feature(self, atom_list, bond_list, atom_degree_list, bond_degree_list, batch_size, mol_length):
        neighbor_feature = []
        for i in range(batch_size):
            atom_degrees = atom_degree_list[i]
            bond_degrees = bond_degree_list[i]
        
            # Ensure indices are within bounds
            atom_degrees = torch.clamp(atom_degrees, 0, mol_length - 1)
            bond_degrees = torch.clamp(bond_degrees, 0, bond_list.size(1) - 1)
        
        # No need to pad, use the original tensors
            atom_features = atom_list[i]
            bond_features = bond_list[i]
        
        # Gather neighbor features
            neighbor_atoms = atom_features[atom_degrees]
            neighbor_bonds = bond_features[bond_degrees]
            
        # Combine atom and bond features
            mol_neighbor_feature = torch.cat([neighbor_atoms, neighbor_bonds], dim=-1)
        
        # Apply neighbor_fc to each atom's neighborhood and sum
            mol_neighbor_feature = self.neighbor_fc(mol_neighbor_feature)
            mol_neighbor_feature = mol_neighbor_feature.sum(dim=1)
        
            neighbor_feature.append(mol_neighbor_feature)
    
        neighbor_feature = torch.stack(neighbor_feature, dim=0)
        return F.relu(neighbor_feature)
    


In [6]:
def create_structured_relative_ecfp_dataframes(df, holdout_size):
    """
    A function to create structured relative ECFP dataframes for training and testing.

    This function processes calixarene data to generate two DataFrames:
    - One for training data where neither host is in the holdout set.
    - One for testing data where one host is in the holdout set for prediction.

    Parameters:
    - holdout_size: Fraction of calixarenes to be used for testing.

    Returns:
    - Tuple of two DataFrames (train_df, test_df) and a list of test hosts
    """
    split_calixarene_dict = {'predictable': ['AP1', 'AP3', 'AP4', 'AP5', 'AP6',
                                    'AP7', 'AP8', 'AP9', 'AH1', 'AH2',
                                    'AH3', 'AH4', 'AH5', 'AH6', 'AH7',
                                    'AM1', 'AM2', 'AO1', 'AO2', 'AO3',
                                    'E1', 'E3', 'E6', 'E7', 'E8', 'E11',
                                    'P-NO2', 'PSC4'],
                    'unpredictable': ['BP0', 'BP1', 'BH2', 'BM1', 'CP1',
                                      'CP2', 'DP2', 'DM1', 'DO2', 'DO3']}
    
    # Read in the .csv file
    calixarene_df = df

    # Determine the holdout calixarenes
    holdout_pred_amount = int(len(split_calixarene_dict['predictable']) * holdout_size)
    holdout_unpred_amount = int(len(split_calixarene_dict['unpredictable']) * holdout_size)

    holdout_calixarenes_pred = random.sample(split_calixarene_dict['predictable'], holdout_pred_amount)
    holdout_calixarenes_unpred = random.sample(split_calixarene_dict['unpredictable'], holdout_unpred_amount)
    all_holdout_calix = holdout_calixarenes_pred + holdout_calixarenes_unpred

    # Initialize lists to hold data for train and test DataFrames
    train_data = []
    test_data = []

    # Target columns for comparison
    target_columns = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    # Set to keep track of test hosts
    test_hosts = set()

    # Iterate over all combinations of two different hosts
    for (idx1, row1), (idx2, row2) in itertools.permutations(calixarene_df.iterrows(), 2):
        host_pair = f"{row1['Host']},{row2['Host']}"
        smiles_pair = f"{row1['cano_smiles']},{row2['cano_smiles']}"
        
        if row1['Host'] not in all_holdout_calix and row2['Host'] not in all_holdout_calix:
            # For training data
            train_row = {'Host_Pair': host_pair, 'SMILES_pair': smiles_pair}
            for target in target_columns:
                train_row[target] = row1[target] - row2[target]
            train_data.append(train_row)
        else:
            if (row1['Host'] in holdout_calixarenes_pred) ^ (row2['Host'] in holdout_calixarenes_pred):
                if row1['Host'] not in holdout_calixarenes_unpred and row2['Host'] not in holdout_calixarenes_unpred:
                    # For test data
                    test_row = {'Host_Pair': host_pair, 'SMILES_pair': smiles_pair}
                    for target in target_columns:
                        test_row[target] = row1[target] - row2[target]
                    test_data.append(test_row)
                    
                    # Add test hosts to the set
                    if row1['Host'] in holdout_calixarenes_pred:
                        test_hosts.add(row1['Host'])
                    if row2['Host'] in holdout_calixarenes_pred:
                        test_hosts.add(row2['Host'])

    # Convert lists to DataFrames
    train_df = pd.DataFrame(train_data)
    test_df = pd.DataFrame(test_data)

    # Convert set of test hosts to a list
    test_hosts_list = list(test_hosts)

    return train_df, test_df, test_hosts_list


def prepare_model_and_data_for_relative_learning(raw_filename, targets=None, p_dropout=0.1, fingerprint_dim=150, 
                                              weight_decay=2, learning_rate=3, radius=3, T=2):
    
    """Prepare data and model for relative learning approach with memory-efficient implementation."""
    """If you are using the GCN model the T is still be taken as an argument bet will not be used in the architecture. Just to make life easier"""
    
    global initial_state 

    
    if targets is None:
        targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

    # Load and preprocess data
    feature_filename = raw_filename.replace('.csv', '.pickle')
    smiles_targets_df = pd.read_csv(raw_filename)
    smilesList = smiles_targets_df.SMILES.values
    
    # Process SMILES strings
    remained_smiles = []
    canonical_smiles_list = []
    for smiles in smilesList:
        try:
            mol = Chem.MolFromSmiles(smiles)
            remained_smiles.append(smiles)
            canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
        except:
            pass
    
    # Filter dataframe and add canonical SMILES
    smiles_targets_df = smiles_targets_df[smiles_targets_df["SMILES"].isin(remained_smiles)]
    smiles_targets_df['cano_smiles'] = canonical_smiles_list

    # Load or generate feature dictionaries
    if os.path.isfile(feature_filename):
        feature_dicts = pickle.load(open(feature_filename, "rb"))
    else:
        feature_dicts = get_smiles_dicts(smilesList)

    # Filter by available features
    remained_df = smiles_targets_df[smiles_targets_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]

    # Create pairs and calculate relative targets more efficiently
    # Use batch processing to avoid memory issues
    batch_size = 32  # Adjust based on available memory
    n_samples = len(remained_df)
    smile_pairs = []
    relative_targets = []
    
    # Process in batches to manage memory
    for i in range(0, n_samples, batch_size):
        batch_end = min(i + batch_size, n_samples)
        batch_df = remained_df.iloc[i:batch_end]
        
        for idx1, row1 in batch_df.iterrows():
            # For each compound in the batch, compare with all other compounds
            # This avoids the n² complexity by processing in smaller chunks
            for idx2, row2 in remained_df.iterrows():
                if idx1 != idx2:  # Don't compare with self
                    smile1 = row1['cano_smiles']
                    smile2 = row2['cano_smiles']
                    target1 = row1[targets].values
                    target2 = row2[targets].values
                    
                    rel_target = target1 - target2
                    smile_pairs.append(f"{smile1},{smile2}")
                    relative_targets.append(rel_target)
                    


    # Create dataframe with pairs and targets
    df = pd.DataFrame(relative_targets, columns=targets)
    df['SMILES_pair'] = smile_pairs
    df = df[['SMILES_pair'] + targets]

    # Get feature dimensions from first pair
    smile1, smile2 = smile_pairs[0].split(',')
    x_atom1, x_bonds1, _, _, _, _ = get_smiles_array([smile1], feature_dicts)
    num_atom_features = x_atom1.shape[-1]
    num_bond_features = x_bonds1.shape[-1]

    # Initialize model and optimizer
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loss_function = nn.MSELoss()
    model = Fingerprint(radius, T, num_atom_features, num_bond_features, fingerprint_dim, len(targets), p_dropout)
    initial_state = {k: v.clone() for k, v in model.state_dict().items()} # This will be used to reinitiate model before each training loop
    model.to(device)
    optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)


    return model, optimizer, loss_function, feature_dicts, df, remained_df


def validate(model, val_df, feature_dicts, loss_function, device, batch_size, return_predictions=False, test_smile=None):
    """Validate model on validation or test data."""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_true_values = []
    result_dict = {}

    with torch.no_grad():
        for i in range(0, len(val_df), batch_size):
            batch_df = val_df.iloc[i:i+batch_size]
            smile_pairs = batch_df.SMILES_pair.values
            y_val = batch_df[targets].values
            
            # Process SMILES pairs
            smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
            
            # Get molecular features
            x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
            x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
            
            # Convert to tensors
            x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
            x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
            x_mask1 = torch.Tensor(x_mask1).to(device)
            
            x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
            x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
            x_mask2 = torch.Tensor(x_mask2).to(device)
            
            # Forward pass
            output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                               x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
            predictions = output_tuple[2]
            
            # Calculate loss
            loss = loss_function(predictions, torch.Tensor(y_val).to(device))
            total_loss += loss.item()
            
            all_predictions.extend(predictions.cpu().numpy())
            all_true_values.extend(y_val)
            
            # Store results for all test data when return_predictions is True
            if return_predictions:
                for j, smile_pair in enumerate(smile_pairs):
                    smile1, smile2 = smile_pair.split(',')
                    
                    # If test_smile is provided, only store data for that specific SMILE
                    if test_smile is None or test_smile in (smile1, smile2):
                        # Use the smile_pair as the key in the result_dict
                        if smile_pair not in result_dict:
                            result_dict[smile_pair] = {}
                        
                        result_dict[smile_pair] = {
                            target: {"actual": y_val[j][k], "predicted": predictions[j][k].item()}
                            for k, target in enumerate(targets)
                        }
    
    avg_loss = total_loss / (len(val_df) // batch_size + 1)
    r2 = r2_score(all_true_values, all_predictions)
    
    if return_predictions:
        return avg_loss, r2, all_true_values, all_predictions, result_dict
    return avg_loss, r2


def train_and_evaluate(model, train_df, test_df, feature_dicts, optimizer, loss_function, num_epochs=800, batch_size=32, patience=30):
    """Train model using provided train and test dataframes with proper data isolation."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    
    # Reset model and optimizer
    model.load_state_dict(initial_state) 
    optimizer = type(optimizer)(model.parameters(), **optimizer.defaults)
    
    # Split training data into train and validation
    train_subset, val_df = train_test_split(train_df, test_size=0.2)
    
    print(f"Training set: {len(train_subset)} samples")
    print(f"Validation set: {len(val_df)} samples")
    print(f"Test set: {len(test_df)} samples")
    
    # Training loop
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        # Train
        model.train()
        total_loss = 0
        train_subset = train_subset.sample(frac=1).reset_index(drop=True)  # Shuffle
        
        # Process in batches to manage memory
        for i in range(0, len(train_subset), batch_size):
            batch_df = train_subset.iloc[i:i+batch_size]
            smile_pairs = batch_df.SMILES_pair.values
            y_val = batch_df[targets].values
            
            # Process SMILES pairs
            smiles_list1, smiles_list2 = zip(*[pair.split(',') for pair in smile_pairs])
            
            # Get molecular features
            x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1, _ = get_smiles_array(smiles_list1, feature_dicts)
            x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2, _ = get_smiles_array(smiles_list2, feature_dicts)
            
            # Convert to tensors
            x_atom1, x_bonds1 = torch.Tensor(x_atom1).to(device), torch.Tensor(x_bonds1).to(device)
            x_atom_index1, x_bond_index1 = torch.LongTensor(x_atom_index1).to(device), torch.LongTensor(x_bond_index1).to(device)
            x_mask1 = torch.Tensor(x_mask1).to(device)
            
            x_atom2, x_bonds2 = torch.Tensor(x_atom2).to(device), torch.Tensor(x_bonds2).to(device)
            x_atom_index2, x_bond_index2 = torch.LongTensor(x_atom_index2).to(device), torch.LongTensor(x_bond_index2).to(device)
            x_mask2 = torch.Tensor(x_mask2).to(device)
            
            # Forward and backward pass
            optimizer.zero_grad()
            output_tuple = model(x_atom1, x_bonds1, x_atom_index1, x_bond_index1, x_mask1,
                               x_atom2, x_bonds2, x_atom_index2, x_bond_index2, x_mask2)
            predictions = output_tuple[2]
            
            loss = loss_function(predictions, torch.Tensor(y_val).to(device))
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / (len(train_subset) // batch_size + 1)
        
        # Validation
        val_loss, val_r2 = validate(model, val_df, feature_dicts, loss_function, device, batch_size)
        print(f'Epoch {epoch+1}/{num_epochs}: Train Loss={avg_loss:.4f}, Val Loss={val_loss:.4f}, Val R2={val_r2:.4f}')
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
    
    # Load best model for evaluation
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    # Evaluate on test set
    test_loss, test_r2, true_values, predictions, result_dict = validate(
        model, test_df, feature_dicts, loss_function, device, batch_size, 
        return_predictions=True
    )
    
    print(f'Final results: Val Loss={best_val_loss:.4f}, Test Loss={test_loss:.4f}, Test R2={test_r2:.4f}')
    
    # Calculate overall metrics
    overall_r2 = r2_score(true_values, predictions)
    print(f'Overall Test R2: {overall_r2:.4f}')
    
    return (best_val_loss, test_loss, test_r2), overall_r2, result_dict


def map_smiles_to_host_pairs(final_dict, remained_df):
    """
    Maps SMILES pairs in the dictionary to host pair names.
    
    Parameters:
    - final_dict: Dictionary with SMILES pairs as keys
    - remained_df: DataFrame with 'Host' and 'cano_smiles' columns
    
    Returns:
    - A new dictionary with host pairs as keys
    """
    # Create a mapping from SMILES to Host
    smiles_to_host = dict(zip(remained_df['cano_smiles'], remained_df['Host']))
    
    # Create the new dictionary with host pairs as keys
    host_pair_dict = {}
    
    for smiles_pair, values in final_dict.items():
        smiles1, smiles2 = smiles_pair.split(',')
        
        # Look up the host names
        host1 = smiles_to_host.get(smiles1, "Unknown")
        host2 = smiles_to_host.get(smiles2, "Unknown")
        
        # Create the host pair key
        host_pair = f"{host1},{host2}"
        
        # Add to the new dictionary
        host_pair_dict[host_pair] = values
    
    return host_pair_dict


def process_and_format_predictions(relative_results, test_hosts, remained_df):
    """
    Process relative learning model results and format the final predictions.
    
    Args:
        relative_results: Dictionary with format {'hostA,hostB': {'target': {'actual': val, 'predicted': val}}}
        test_hosts: List of host names that are being tested
        remained_df: DataFrame with 'Host' column and individual values for each host
        
    Returns:
        Dictionary with final predictions and actual values for each test host and target
    """
    # Initialize result dictionary
    final_predictions = {host: {} for host in test_hosts}
    
    # Initialize counters to calculate averages later
    count_dict = {host: {} for host in test_hosts}
    
    # Process each host pair
    for pair, targets in relative_results.items():
        host1, host2 = pair.split(',')
        
        for target, values in targets.items():
            predicted_relative = values['predicted']
            
            # If host1 is a test host
            if host1 in test_hosts:
                # Get the value for host2 from the dataframe
                host2_value = remained_df[remained_df['Host'] == host2][target].values[0]
                
                # Calculate predicted value for host1
                host1_predicted = predicted_relative + host2_value
                
                # Add to sum for averaging later
                if target not in final_predictions[host1]:
                    final_predictions[host1][target] = host1_predicted
                    count_dict[host1][target] = 1
                else:
                    final_predictions[host1][target] += host1_predicted
                    count_dict[host1][target] += 1
            
            # If host2 is a test host
            if host2 in test_hosts:
                # Get the value for host1 from the dataframe
                host1_value = remained_df[remained_df['Host'] == host1][target].values[0]
                
                # Calculate predicted value for host2
                host2_predicted = host1_value - predicted_relative
                
                # Add to sum for averaging later
                if target not in final_predictions[host2]:
                    final_predictions[host2][target] = host2_predicted
                    count_dict[host2][target] = 1
                else:
                    final_predictions[host2][target] += host2_predicted
                    count_dict[host2][target] += 1
    
    # Calculate averages and format the output
    formatted_predictions = {}
    for host in test_hosts:
        formatted_predictions[host] = []
        for target in final_predictions[host]:
            # Calculate average prediction
            avg_prediction = final_predictions[host][target] / count_dict[host][target]
            
            # Get the actual value for the test host from the dataframe
            actual_value = remained_df[remained_df['Host'] == host][target].values[0]
            
            # Append the tuple (avg_prediction, actual_value) to the list for the host
            formatted_predictions[host].append((avg_prediction, actual_value))
    
    return formatted_predictions


def run_multiple_iterations(model, remained_df, feature_dicts, optimizer, loss_function, num_iterations=20, holdout_size=0.10, num_epochs=5, batch_size=32, patience=30):

    
    # Initialize results dictionary
    all_results = {}
    
    # Run the workflow for the specified number of iterations
    for iteration in range(num_iterations):
        print(f"Running iteration {iteration+1}/{num_iterations}")
        
        # Step 1: Create train and test dataframes
        train_df, test_df, test_hosts = create_structured_relative_ecfp_dataframes(remained_df, holdout_size)
        
        # Step 2: Train and evaluate model
        fold_results, overall_r2, final_dic = train_and_evaluate(
            model, train_df, test_df, feature_dicts, optimizer,
            loss_function, num_epochs=num_epochs, batch_size=batch_size, patience=patience
        )
        
        # Step 3: Map SMILES to host pairs
        host_pair_dict = map_smiles_to_host_pairs(final_dic, remained_df)
        
        # Step 4: Process and format predictions
        new_final = process_and_format_predictions(host_pair_dict, test_hosts, remained_df)
        
        # Store the results for this iteration
        all_results[iteration] = new_final
        
        print(f"Iteration {iteration+1} completed with overall R² = {overall_r2:.4f}")
        
    return all_results


In [7]:
# ________________Single loop for testing ____________________


# # Define targets
# targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']

# # Prepare model and data
# model, optimizer, loss_function, feature_dicts, df , remained_df= prepare_model_and_data_for_relative_learning(
#     '/notebooks/Codebase/Database/cal_abs.csv'
# )

# train_df, test_df, test_hosts= create_structured_relative_ecfp_dataframes(remained_df, holdout_size= 0.10)

# # Train and evaluate model
# fold_results, overall_r2, final_dic = train_and_evaluate(model, train_df, test_df, feature_dicts, optimizer,
#                                                          loss_function, num_epochs=800, batch_size=32, patience=30)

# host_pair_dict = map_smiles_to_host_pairs(final_dic, remained_df)
# new_final= process_and_format_predictions(host_pair_dict, test_hosts, remained_df)

In [8]:
targets = ['H3K4', 'H3K4ac', 'H3K4me1', 'H3K4me2', 'H3K4me3', 'H3K9me3', 'H3R2me2a', 'H3R2me2s']
holdout_size= [0.05,0.1,0.15,0.25,0.5,0.75]
model, optimizer, loss_function, feature_dicts, df , remained_df= prepare_model_and_data_for_relative_learning(
    '/notebooks/Codebase/Database/cal_abs.csv')

    
for i in holdout_size:

    all_iteration_results = run_multiple_iterations(
        model, 
        remained_df, 
        feature_dicts, 
        optimizer, 
        loss_function, 
        num_iterations=20, 
        holdout_size=i, 
        num_epochs=800, 
        batch_size=32, 
        patience=30
    )
    output = open(f'/notebooks/Codebase/Result_dict/20 split {i} AttentiveFP Relative_val.pkl', 'wb')
    pickle.dump(all_iteration_results, output)
    output.close()
 



Running iteration 1/20
Training set: 1065 samples
Validation set: 267 samples
Test set: 74 samples
Epoch 1/800: Train Loss=6.4317, Val Loss=6.4255, Val R2=0.0295
Epoch 2/800: Train Loss=5.9423, Val Loss=5.8535, Val R2=0.1205
Epoch 3/800: Train Loss=5.1001, Val Loss=5.0910, Val R2=0.2625
Epoch 4/800: Train Loss=4.3081, Val Loss=4.6556, Val R2=0.3364
Epoch 5/800: Train Loss=3.7744, Val Loss=4.1392, Val R2=0.4200
Epoch 6/800: Train Loss=3.3808, Val Loss=3.7536, Val R2=0.4658
Epoch 7/800: Train Loss=3.3764, Val Loss=3.5034, Val R2=0.5163
Epoch 8/800: Train Loss=2.9064, Val Loss=3.2031, Val R2=0.5477
Epoch 9/800: Train Loss=2.6288, Val Loss=2.6072, Val R2=0.6258
Epoch 10/800: Train Loss=2.3493, Val Loss=2.4985, Val R2=0.6379
Epoch 11/800: Train Loss=2.2285, Val Loss=2.0165, Val R2=0.7065
Epoch 12/800: Train Loss=1.7656, Val Loss=1.7794, Val R2=0.7525
Epoch 13/800: Train Loss=1.7090, Val Loss=1.5840, Val R2=0.7736
Epoch 14/800: Train Loss=1.4677, Val Loss=1.3617, Val R2=0.8106
Epoch 15/800: 

In [23]:
 all_iteration_results


{0: {'AP5': [(0.05550264146527041, 1.458615023),
   (0.5390959472220614, 2.041220329),
   (-0.838949825673148, 0.262364264),
   (-2.57949781512927, -0.820980552),
   (-2.56157370163682, -1.966112856),
   (-1.6889804674950566, -1.171182982),
   (-1.2614553652631024, -0.510825624),
   (-0.5560262982326047, 0.470003629)],
  'E1': [(1.6512020360926112, 0.875468737),
   (2.378945521128041, 1.589235205),
   (0.44231392177894485, -0.235722334),
   (-1.083037928364961, -1.386294361),
   (-1.2985057499049044, -2.207274913),
   (-0.33718957122410065, -1.237874356),
   (0.034410605560279736, -0.967584026),
   (0.8411384319362742, 0.470003629)]},
 1: {'E7': [(2.175977416514975, 2.066862759),
   (3.187662330449677, 2.48490665),
   (0.9632762861739562, 1.435084525),
   (-0.4388801206515948, -0.415515444),
   (-0.8752567773263221, -0.248461359),
   (0.2271718796288217, 0.693147181),
   (0.5455148444233098, 1.193922468),
   (1.4410982527785772, 2.186051277)],
  'AH1': [(0.959515008282605, 0.262364264)

In [10]:
with open('/notebooks/Codebase/Result_dict/20 split 0.1 GCN Relative_val.pkl', 'wb') as fp:
    pickle.dump(all_iteration_results, fp)
    print('dictionary saved successfully to file')

dictionary saved successfully to file


In [9]:
all_iteration_results

{0: {'AH1': [(-0.16032774611179215, 0.262364264),
   (0.5488755563785035, 2.48490665),
   (-1.3009706865561281, -1.021651248),
   (-3.089521930432129, -3.473768074),
   (-2.845435843045211, -3.352407217),
   (-1.6999259419270067, -2.375155786),
   (-1.4687304494822013, -1.832581464),
   (-0.9413323829758977, -0.274436846)]},
 1: {'AP3': [(1.2204013131557383, 0.741937345),
   (1.7471130521063751, 1.791759469),
   (0.1569280691354378, 0.182321557),
   (-1.3737707085654869, -1.171182982),
   (-1.5636460387296254, -1.660731207),
   (-0.5977129032602402, -0.867500568),
   (-0.15442258574774126, -0.430782916),
   (0.5382778023291911, 0.587786665)]},
 2: {'AH5': [(3.292203472486188, 3.135494216),
   (2.5998532460077404, 4.605170186),
   (1.214535629590268, 2.116255515),
   (-0.41999407166714614, -0.105360516),
   (-0.11863407902588266, 0.0),
   (0.9192535979974842, 1.223775432),
   (0.6826177640192131, 1.335001067),
   (1.63558171364921, 2.197224577)]},
 3: {'AP4': [(0.4972313270321549, 1.335

In [14]:
ttest= pd.read_pickle('/notebooks/Codebase/Result_dict/20 split 0.5 AttentiveFP Relative_val.pkl')


In [15]:
ttest

{0: {'AO3': [(0.10073155780952468, 1.458615023),
   (1.137128266833657, 1.824549292),
   (-1.5584364082029893, 0.78845736),
   (-3.9511925743060106, -0.562118918),
   (-2.839211883177061, -1.108662625),
   (-1.6841191726788545, -0.510825624),
   (-1.7373295204696642, 0.470003629),
   (-1.0693111139332456, 0.587786665)],
  'AP6': [(0.6771811620243309, 0.470003629),
   (0.4300827101503679, -0.127833372),
   (-0.6720044621759458, -2.120263536),
   (-3.1951645485518227, -4.342805922),
   (-2.7667212521870677, -4.199705078),
   (-4.18363266623558, -3.244193633),
   (-2.214786028183836, -3.015934981),
   (-0.32072128640119935, -1.46967597)],
  'AP1': [(0.0712172685935333, 1.16315081),
   (1.0787905670156732, 2.186051277),
   (-1.7991605309919425, 0.0),
   (-4.49153802977685, -1.30933332),
   (-3.269817588783959, -2.302585093),
   (-2.686885196804104, -1.272965676),
   (-2.4378449793201495, -0.820980552),
   (-1.3460239381806307, 0.405465108)],
  'E1': [(0.4638569829357322, 0.875468737),
   (